# PubMedQA Evidence Classification (yes / no / maybe)**Author:** Anshika Reddy  Classify whether PubMed abstract evidence supports answering a biomedical research question with **yes**, **no**, or **maybe**.- Dataset: expert-labelled `pqa_labeled` only ([qiaojin/PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA)) — **not** the 211k artificial split.- Input: question + abstract- Models: TF-IDF + Logistic Regression, BERT-base, PubMedBERT

## 0. Setup (Colab-friendly)

In [ ]:
# Optional: run on Google Colab with GPU (Runtime -> Change runtime type)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_COLAB


In [ ]:
%pip install -q datasets transformers torch scikit-learn pandas matplotlib seaborn accelerate tqdm requests

import os, sys, json, time
from pathlib import Path

# Point to repo root (adjust if notebook lives elsewhere)
if IN_COLAB:
    # Upload or git-clone the project, then:
    # !git clone https://github.com/YOUR_USER/pubmedqa-evidence-classification.git
    ROOT = Path.cwd() / "pubmedqa-evidence-classification"
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
ROOT


In [ ]:
from src.config import RunConfig, BIOMED_MODEL_NAME, GENERAL_MODEL_NAME, LABELS
from src.utils import set_seed

SEED = 42
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"  # set SMOKE_TEST=1 for CPU smoke run

config = RunConfig(seed=SEED, smoke_test=SMOKE_TEST)
set_seed(config.seed)
config


## 1. Load data & official split

In [ ]:
from collections import Counter
import pandas as pd
from src.data_loading import load_train_test, build_input_text

train_df, test_df = load_train_test(config)
print(f"Train pool: {len(train_df)} | Official test: {len(test_df)}")
print("Train labels:", Counter(train_df["label"]))
print("Test labels:", Counter(test_df["label"]))

train_df.head(2)


In [ ]:
# Validation split from official train pool (for early stopping)
val_df = train_df.sample(frac=0.15, random_state=SEED)
train_fit_df = train_df.drop(val_df.index)
len(train_fit_df), len(val_df), len(test_df)


In [ ]:
# Inspect one example
row = train_df.iloc[0]
print("Question:", row["question"][:120], "...")
print("Abstract snippet:", row["abstract"][:200], "...")
print("Label:", row["label"])


## 2. Baseline — TF-IDF + Logistic Regression

In [ ]:
from src.train_baseline import train_baseline, predict_baseline
from src.evaluate import compute_metrics, ids_to_labels, export_run_results, plot_confusion_matrix, maybe_class_error_analysis
from src.config import LABEL2ID

pipe, baseline_meta = train_baseline(train_fit_df, config)
pred_ids, _ = predict_baseline(pipe, test_df["text"].tolist())
baseline_preds = ids_to_labels(pred_ids)
baseline_metrics = compute_metrics(test_df["label"], baseline_preds)
baseline_metrics


In [ ]:
export_run_results("baseline_tfidf_lr", baseline_metrics, config, extra={"training": baseline_meta})
yt = [LABEL2ID[x] for x in test_df["label"]]
yp = [LABEL2ID[x] for x in baseline_preds]
plot_confusion_matrix(yt, yp, config.results_dir / "baseline_tfidf_lr_confusion_matrix.png", title="Baseline")
maybe_class_error_analysis(test_df, baseline_preds, config.results_dir / "baseline_tfidf_lr_maybe_errors.csv")


## 3. Fine-tune BERT-base-uncased

In [ ]:
import torch
from src.train_transformer import train_transformer, load_transformer_for_inference, predict_transformer

device_note = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device_note)

bert_dir = config.checkpoint_dir / "bert_base"
trainer_bert, bert_meta = train_transformer(
    GENERAL_MODEL_NAME,
    train_fit_df,
    val_df,
    config,
    bert_dir,
    "bert_base",
)
bert_model, bert_tok, device = load_transformer_for_inference(bert_dir / "best")
bert_pred_ids, _ = predict_transformer(bert_model, bert_tok, device, test_df["text"].tolist(), config.max_length)
bert_preds = ids_to_labels(bert_pred_ids)
bert_metrics = compute_metrics(test_df["label"], bert_preds)
export_run_results("bert_base", bert_metrics, config, extra={"training": bert_meta})
bert_metrics


## 4. Fine-tune PubMedBERT (biomedical)

In [ ]:
pubmed_dir = config.checkpoint_dir / "pubmedbert"
trainer_pub, pub_meta = train_transformer(
    BIOMED_MODEL_NAME,
    train_fit_df,
    val_df,
    config,
    pubmed_dir,
    "pubmedbert",
)
pub_model, pub_tok, device = load_transformer_for_inference(pubmed_dir / "best")
pub_pred_ids, _ = predict_transformer(pub_model, pub_tok, device, test_df["text"].tolist(), config.max_length)
pub_preds = ids_to_labels(pub_pred_ids)
pub_metrics = compute_metrics(test_df["label"], pub_preds)
export_run_results("pubmedbert", pub_metrics, config, extra={"training": pub_meta})
pub_metrics


## 5. Evaluation summary & maybe-class error analysis

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

for name, preds in [
    ("baseline", baseline_preds),
    ("bert_base", bert_preds),
    ("pubmedbert", pub_preds),
]:
    print("=" * 60, name)
    print(classification_report(test_df["label"], preds, labels=LABELS, zero_division=0))

# Compare maybe F1 across models
rows = []
for name, m in [("baseline", baseline_metrics), ("bert_base", bert_metrics), ("pubmedbert", pub_metrics)]:
    rep = m["classification_report"]
    rows.append({"model": name, "accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "maybe_f1": rep["maybe"]["f1-score"]})
summary = pd.DataFrame(rows)
summary.to_csv(config.results_dir / "notebook_model_comparison.csv", index=False)
summary


In [ ]:
# Examples where gold label is maybe but model disagrees (PubMedBERT)
errors = test_df.copy()
errors["predicted"] = pub_preds
errors.loc[(errors["label"] == "maybe") & (errors["predicted"] != "maybe"), ["question", "label", "predicted"]].head(10)


## 6. Training time & inference latency

In [ ]:
from src.evaluate import measure_inference_latency

def baseline_fn(texts):
    return predict_baseline(pipe, texts)[0]

latency_rows = []
latency_rows.append({"model": "baseline", **measure_inference_latency(baseline_fn, test_df["text"].tolist())})

def make_tf_predict(model, tok, dev):
    def fn(texts):
        return predict_transformer(model, tok, dev, texts, config.max_length, batch_size=1)[0]
    return fn

latency_rows.append({"model": "bert_base", **measure_inference_latency(make_tf_predict(bert_model, bert_tok, device), test_df["text"].tolist())})
latency_rows.append({"model": "pubmedbert", **measure_inference_latency(make_tf_predict(pub_model, pub_tok, device), test_df["text"].tolist())})

timing = pd.DataFrame(latency_rows)
timing["train_seconds"] = [
    baseline_meta.get("training", {}).get("elapsed_seconds"),
    bert_meta.get("training", {}).get("elapsed_seconds"),
    pub_meta.get("training", {}).get("elapsed_seconds"),
]
timing.to_csv(config.results_dir / "timing_comparison.csv", index=False)
timing


## 7. Reproducibility notes- Set `SMOKE_TEST=1` before running for a CPU-friendly subsample (`config.smoke_test`).- Transformer checkpoints save under `checkpoints/`; re-run cells to **resume** if `checkpoint-*` exists.- Full metrics exported to `results/*.json` and `results/*.csv`.